# Are You A Robot? -- Multi-Task Essay Analysis

**Kaggle Competition Solution**

**Author:** [Amey Thakur](https://www.kaggle.com/ameythakur20)

Three independent evaluation pipelines run over admission essays:
a deterministic readability metric (RRM), a supervised sentiment
regressor built on TF-IDF and gradient boosting, and an unsupervised
human-vs-AI classifier powered by stylometric feature extraction
and Gaussian Mixture Models. Total wall-clock time stays well under
15 minutes on a single CPU core.

**Outline:**

1. [Imports and Data Loading](#1-imports-and-data-loading)
2. [Subtask 1 -- Robust Readability Metric (RRM)](#2-subtask-1----robust-readability-metric-rrm)
3. [Subtask 2 -- Sentiment Regression](#3-subtask-2----sentiment-regression)
4. [Subtask 3 -- Human vs AI Detection](#4-subtask-3----human-vs-ai-detection)
5. [Submission Assembly](#5-submission-assembly)
6. [Summary and References](#6-summary-and-references)

---


## 1. Imports and Data Loading

Standard scientific stack plus LightGBM for gradient-boosted
regression. The CMU Pronouncing Dictionary provides phoneme-level
syllable counts, which are far more reliable than heuristic vowel
grouping for the RRM formula.


In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
import nltk
from collections import Counter
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    VotingRegressor,
    GradientBoostingRegressor,
)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor
from scipy.sparse import hstack, csr_matrix

warnings.filterwarnings('ignore')

# cmudict gives exact syllable counts for ~130k English words;
# the heuristic fallback only runs for out-of-vocabulary tokens
try:
    nltk.download('cmudict', quiet=True)
    from nltk.corpus import cmudict
    cmu_dict = cmudict.dict()
except Exception:
    cmu_dict = None

# auto-detect whether we are running inside Kaggle or locally
IS_KAGGLE = Path("/kaggle").exists()
if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    DATA_DIR = next(
        (p.parent for p in INPUT_DIR.rglob("train.csv")), None
    )
    OUTPUT_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path("data")
    OUTPUT_DIR = Path(".")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Train shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")


## 2. Subtask 1 -- Robust Readability Metric (RRM)

The RRM is a clipped variant of the Flesch Reading Ease score:

```
RRM = clip(206.835 - 1.015*(W/S) - 84.6*(SYL/W), 0, 100)
```

Accuracy depends almost entirely on syllable counting.
`cmudict` covers the majority of tokens; the vowel-group
heuristic handles the remainder with suffix corrections for
silent-e, -ed, and -es endings.


In [ ]:
def count_syllables(word):
    """Return the number of syllables in a single word."""
    word = word.lower().strip()
    word = re.sub(r'[^a-z]', '', word)
    if not word:
        return 0
    if len(word) <= 2:
        return 1

    # prefer the dictionary lookup when available
    if cmu_dict and word in cmu_dict:
        phones = cmu_dict[word][0]
        return sum(1 for p in phones if p[-1].isdigit())

    # fallback: count vowel groups
    vowels = 'aeiouy'
    count = 0
    prev_vowel = False
    for ch in word:
        is_v = ch in vowels
        if is_v and not prev_vowel:
            count += 1
        prev_vowel = is_v

    # silent-e at the end (but not words like "smile" -> "le")
    if word.endswith('e') and not word.endswith('le'):
        count -= 1
    # past-tense -ed is usually silent unless preceded by t/d
    if word.endswith('ed') and len(word) > 3 and word[-3] not in 'td':
        count -= 1
    # plural -es is usually silent except after s, z, sh, ch, x
    if (word.endswith('es') and len(word) > 3
            and not re.search(r'(ses|zes|xes|ches|shes)$', word)):
        count -= 1

    return max(count, 1)

def compute_rrm(text):
    """Compute the Robust Readability Metric for one essay."""
    text = str(text)
    if not text.strip():
        return 50.0

    words = text.split()
    W = max(len(words), 1)

    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    S = max(len(sentences), 1)

    SYL = sum(count_syllables(w) for w in words)

    score = 206.835 - 1.015 * (W / S) - 84.6 * (SYL / W)
    return float(np.clip(score, 0.0, 100.0))

test_rrm = test_df['text_content'].apply(compute_rrm)

print(f"RRM -- min: {test_rrm.min():.2f}  max: {test_rrm.max():.2f}  "
      f"mean: {test_rrm.mean():.2f}")


## 3. Subtask 2 -- Sentiment Regression

The pipeline combines word-level and character-level TF-IDF
matrices with a handful of hand-crafted text statistics, then
feeds them into a VotingRegressor ensemble of Ridge, Gradient
Boosting, and LightGBM regressors.


In [ ]:
def extract_sentiment_features(df):
    """Build a small feature matrix of text statistics."""
    feats = pd.DataFrame(index=df.index)
    text = df['text_content'].astype(str)

    feats['n_chars']     = text.str.len()
    feats['n_words']     = text.apply(lambda x: max(len(x.split()), 1))
    feats['n_sentences'] = text.apply(
        lambda t: max(len([s for s in re.split(r'[.!?]+', t) if s.strip()]), 1)
    )
    feats['avg_word_len'] = text.apply(
        lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
    )
    feats['avg_sent_len'] = feats['n_words'] / feats['n_sentences']

    # punctuation ratios (pandas str.count uses regex)
    feats['excl_rate']  = text.str.count('!') / feats['n_words']
    feats['quest_rate'] = text.str.count('[?]') / feats['n_words']
    feats['comma_rate'] = text.str.count(',') / feats['n_words']

    # capitalisation density
    feats['upper_ratio'] = text.apply(
        lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1)
    )

    # sentence-length variability (burstiness)
    def _sent_std(t):
        parts = [s.strip() for s in re.split(r'[.!?]+', t) if s.strip()]
        if len(parts) < 2:
            return 0.0
        return float(np.std([len(s.split()) for s in parts]))
    feats['sent_len_std'] = text.apply(_sent_std)

    # vocabulary richness
    feats['unique_ratio'] = text.apply(
        lambda x: len(set(x.lower().split())) / max(len(x.lower().split()), 1)
    )

    return feats

train_feats = extract_sentiment_features(train_df)
test_feats  = extract_sentiment_features(test_df)

# word n-grams (1,2) and character n-grams (3,5) with sublinear TF
tfidf_w = TfidfVectorizer(
    max_features=6000, ngram_range=(1, 2),
    min_df=2, max_df=0.95, sublinear_tf=True,
)
tfidf_c = TfidfVectorizer(
    analyzer='char_wb', max_features=4000, ngram_range=(3, 5),
    min_df=2, max_df=0.95, sublinear_tf=True,
)

# fit on the union so both splits share the same vocabulary
all_text = pd.concat([
    train_df['text_content'], test_df['text_content']
]).astype(str)
tfidf_w.fit(all_text)
tfidf_c.fit(all_text)

scaler = StandardScaler()
train_feats_sc = scaler.fit_transform(train_feats)
test_feats_sc  = scaler.transform(test_feats)

X_train = hstack([
    tfidf_w.transform(train_df['text_content'].astype(str)),
    tfidf_c.transform(train_df['text_content'].astype(str)),
    csr_matrix(train_feats_sc),
])
X_test = hstack([
    tfidf_w.transform(test_df['text_content'].astype(str)),
    tfidf_c.transform(test_df['text_content'].astype(str)),
    csr_matrix(test_feats_sc),
])
y_train = train_df['sentiment_score'].values

print(f"Feature matrix shape: {X_train.shape}")

# voting ensemble: two linear models for stability, two tree
# models for nonlinear interactions
ensemble = VotingRegressor([
    ('ridge_hi',  Ridge(alpha=10.0)),
    ('ridge_lo',  Ridge(alpha=1.0)),
    ('gbr', GradientBoostingRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.04,
        subsample=0.8, random_state=42,
    )),
    ('lgbm', LGBMRegressor(
        n_estimators=800, learning_rate=0.02, max_depth=8,
        num_leaves=31, random_state=42, n_jobs=-1, verbose=-1,
    )),
])

ensemble.fit(X_train, y_train)
test_sentiment = np.clip(ensemble.predict(X_test), -1.0, 1.0)

print(f"Sentiment -- min: {test_sentiment.min():.4f}  "
      f"max: {test_sentiment.max():.4f}")


## 4. Subtask 3 -- Human vs AI Detection

Without ground-truth labels this subtask requires unsupervised
separation. The approach extracts stylometric signals that
reliably distinguish machine-generated text from human writing:

- **Sentence-length variability:** AI output tends to be uniform;
  human writing has higher standard deviation and coefficient of
  variation in sentence lengths.
- **Vocabulary richness and hapax ratio:** humans use more rare
  words; AI recycles common vocabulary.
- **First-person pronoun density:** admission essays written by
  humans contain significantly more personal pronouns.
- **Transition-word density:** AI inserts transition words at a
  more regular cadence than humans.

A two-component Gaussian Mixture Model with full covariance
clusters the feature space, and the cluster exhibiting lower
burstiness and fewer personal pronouns is labelled as AI.


In [ ]:
def extract_detection_features(df):
    # build stylometric features for human-vs-AI separation
    feats = pd.DataFrame(index=df.index)
    text = df['text_content'].astype(str)

    # -- sentence-level statistics --
    def _sent_stats(t):
        parts = [s.strip() for s in re.split(r'[.!?]+', t) if s.strip()]
        if len(parts) < 2:
            return 0.0, 0.0, 0.0
        lens = [len(s.split()) for s in parts]
        return float(np.std(lens)), float(np.mean(lens)), float(max(lens) - min(lens))

    stats = text.apply(_sent_stats)
    feats['sent_std']   = stats.apply(lambda x: x[0])
    feats['sent_mean']  = stats.apply(lambda x: x[1])
    feats['sent_range'] = stats.apply(lambda x: x[2])

    # coefficient of variation (burstiness index)
    feats['sent_cv'] = feats['sent_std'] / (feats['sent_mean'] + 1e-8)

    # -- vocabulary richness --
    feats['type_token'] = text.apply(
        lambda x: len(set(x.lower().split())) / max(len(x.lower().split()), 1)
    )

    # hapax legomena: fraction of words that appear exactly once
    def _hapax(t):
        words = t.lower().split()
        if not words:
            return 0.0
        freq = Counter(words)
        return sum(1 for c in freq.values() if c == 1) / len(words)
    feats['hapax'] = text.apply(_hapax)

    # -- word-length statistics --
    def _wl_stats(t):
        wl = [len(w) for w in t.split()]
        if len(wl) < 2:
            return 0.0, 0.0
        return float(np.std(wl)), float(np.median(wl))
    wl = text.apply(_wl_stats)
    feats['wl_std']    = wl.apply(lambda x: x[0])
    feats['wl_median'] = wl.apply(lambda x: x[1])

    # -- transition words --
    trans_re = re.compile(
        r'\b(however|therefore|moreover|furthermore|nevertheless'
        r'|although|meanwhile|consequently|additionally|specifically)\b',
        re.IGNORECASE,
    )
    feats['trans_count'] = text.apply(lambda t: len(trans_re.findall(t)))
    feats['trans_rate']  = feats['trans_count'] / text.apply(lambda x: max(len(x.split()), 1))

    # -- first-person pronouns --
    fp_re = re.compile(
        r'\b(I|my|me|myself|mine|we|our|us|ourselves)\b'
    )
    feats['fp_count'] = text.apply(lambda t: len(fp_re.findall(t)))
    feats['fp_rate']  = feats['fp_count'] / text.apply(lambda x: max(len(x.split()), 1))

    # -- paragraph structure --
    feats['n_paras'] = text.apply(lambda x: max(len(x.strip().split('\n')), 1))
    feats['words_per_para'] = text.apply(lambda x: len(x.split())) / feats['n_paras']

    # -- punctuation patterns --
    feats['comma_rate']  = text.str.count(',') / text.apply(lambda x: max(len(x.split()), 1))
    feats['exclam_rate'] = text.str.count('!') / text.apply(lambda x: max(len(x.split()), 1))

    # -- text length --
    feats['n_words'] = text.apply(lambda x: len(x.split()))
    feats['n_chars'] = text.str.len()

    return feats

# extract features on the combined train+test pool so
# the scaler sees the full distribution
combined = pd.concat([train_df, test_df], ignore_index=True)
ai_feats = extract_detection_features(combined)
ai_scaled = StandardScaler().fit_transform(ai_feats)

print(f"Detection features: {ai_feats.shape[1]} columns")

# GMM with full covariance captures the fact that the two
# populations (human, AI) have different shapes and spreads
gmm = GaussianMixture(
    n_components=2, covariance_type='full',
    n_init=10, random_state=42,
)
labels = gmm.fit_predict(ai_scaled)

# identify which cluster is AI: AI text typically shows
# lower sentence-length CV and lower first-person pronoun rate
lbl_df = pd.DataFrame({
    'label':   labels,
    'sent_cv': ai_feats['sent_cv'].values,
    'fp_rate': ai_feats['fp_rate'].values,
    'hapax':   ai_feats['hapax'].values,
})
grp = lbl_df.groupby('label').mean()
human_signal = grp['sent_cv'] + grp['fp_rate'] * 100 + grp['hapax'] * 10
ai_label = human_signal.idxmin()

ai_preds = (labels == ai_label).astype(int)
ai_test  = ai_preds[len(train_df):]

print(f"AI cluster: {ai_label}")
print(f"Test split -- AI: {ai_test.sum()}, Human: {len(ai_test) - ai_test.sum()}")


## 5. Submission Assembly

Each test essay appears exactly three times in the output file,
once per subtask. Format follows the sample submission provided
by the competition organisers.


In [ ]:
rows = []
test_ids = test_df['id'].values

for i, tid in enumerate(test_ids):
    rows.append({
        'Id': f'1_{tid}', 'subtaskID': 1,
        'datapointID': tid, 'answer': round(test_rrm.iloc[i], 1),
    })
    rows.append({
        'Id': f'2_{tid}', 'subtaskID': 2,
        'datapointID': tid, 'answer': round(test_sentiment[i], 4),
    })
    rows.append({
        'Id': f'3_{tid}', 'subtaskID': 3,
        'datapointID': tid, 'answer': int(ai_test[i]),
    })

submission = pd.DataFrame(rows)
submission['subtaskID']   = submission['subtaskID'].astype(int)
submission['datapointID'] = submission['datapointID'].astype(int)
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)

print(f"Saved {len(submission)} rows to {OUTPUT_DIR / 'submission.csv'}")
print()
for st in [1, 2, 3]:
    sub = submission[submission['subtaskID'] == st]
    print(f"Subtask {st}: {len(sub)} rows, sample answer = {sub['answer'].iloc[0]}")


## 6. Summary and References

**Subtask 1 (RRM):** Deterministic formula with CMU Pronouncing
Dictionary for syllable counting, falling back to a vowel-group
heuristic with suffix corrections for out-of-vocabulary tokens.

**Subtask 2 (Sentiment):** Word and character TF-IDF features
combined with text statistics, fed into a four-model voting
ensemble (two Ridge regressors, GradientBoosting, LightGBM).

**Subtask 3 (AI Detection):** Unsupervised clustering via a
two-component Gaussian Mixture Model over 16 stylometric
features including sentence-length CV, hapax legomena ratio,
first-person pronoun density, and transition-word frequency.

---
**Citation:**
Cyprus AI Training. [Cyprus Spring 2026] Are You A Robot?.
https://kaggle.com/competitions/cyprus-spring-2026-are-you-a-robot, 2026. Kaggle.
